In [1]:
import json
import pandas as pd
import numpy as np
import pickle
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate, StratifiedShuffleSplit
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import precision_score, recall_score, roc_auc_score, accuracy_score, f1_score

In [2]:
def cross_validation_binary_tree(model_id, model_type):
    with open(f'results/{model_id}_{model_type}_best_within_one.json', 'r') as f:
        params = json.load(f)
    f.close()
    del params["achieved_value"]
    int_param_list = ["max_depth", "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf", "max_leaf_nodes"]
    for key, value in params.items():
        if key in int_param_list:
            params[key] = int(value)
        else:
            params[key] = value
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    # define metric
    metric_list = ['precision', "recall", "roc_auc", "accuracy", "f1"]
    # set model
    model = DecisionTreeClassifier(**params)
    X = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()
    cv_test = cross_validate(model, X, y, cv=cv, scoring=metric_list)
    # print result
    print(model_id, model_type)
    print("Recall: ", cv_test["test_recall"].mean().round(3))
    print("Roc: ", cv_test["test_roc_auc"].mean().round(3))
    print("Accuracy: ", cv_test["test_accuracy"].mean().round(3))
    # save results
    with open(f"results/{model_id}_{model_type}.pkl", "wb") as f:
        pickle.dump(cv_test, f)

In [3]:
cross_validation_binary_tree("model_1","binary_tree")

model_1 binary_tree
Recall:  0.76
Roc:  0.848
Accuracy:  0.772


In [4]:
cross_validation_binary_tree("model_2","binary_tree")

model_2 binary_tree
Recall:  0.845
Roc:  0.786
Accuracy:  0.741


In [5]:
def cross_validation_binary_tree_us(model_id, model_type):
    with open(f'results/{model_id}_{model_type}_best_within_one.json', 'r') as f:
        params = json.load(f)
    f.close()
    del params["achieved_value"]
    int_param_list = ["max_depth", "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf", "max_leaf_nodes"]
    for key, value in params.items():
        if key in int_param_list:
            params[key] = int(value)
        else:
            params[key] = value
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    # define metric
    metric_list = ['precision', "recall", "roc_auc", "accuracy", "f1"]
    # set model
    model = DecisionTreeClassifier(**params)
    X = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()
    cv_test = {
        "fold": [],
        'test_precision': [],
        'test_recall': [],
        'test_roc_auc': [],
        'test_accuracy': [],
        'test_f1': []
    }
    fold_indices = list(cv.split(X, y))
    data_balancer = RandomOverSampler()
    for current_fold in range(len(fold_indices)):
        cv_test['fold'].append(current_fold)
        train_idx, val_idx = fold_indices[current_fold]
        X_train, y_train = X.iloc[train_idx], y[train_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]
        X_train_balanced, y_train_balanced = data_balancer.fit_resample(X_train, y_train)
        fitted_model = model.fit(X_train_balanced, y_train_balanced)
        y_pred = fitted_model.predict(X_val)
        y_pred_proba = fitted_model.predict_proba(X_val)
        cv_test['test_precision'].append(precision_score(y_val, y_pred))
        cv_test['test_recall'].append(recall_score(y_val, y_pred))
        cv_test['test_roc_auc'].append(roc_auc_score(y_val, y_pred_proba[:, 1]))
        cv_test['test_accuracy'].append(accuracy_score(y_val, y_pred))
        cv_test['test_f1'].append(f1_score(y_val, y_pred))
    for metric in ['test_precision', 'test_recall', 'test_roc_auc', 'test_accuracy', 'test_f1']:
        cv_test[metric] = np.array(cv_test[metric])

    # print result
    print(model_id, model_type)
    print("Recall: ", cv_test["test_recall"].mean().round(3))
    print("Roc: ", cv_test["test_roc_auc"].mean().round(3))
    print("Accuracy: ", cv_test["test_accuracy"].mean().round(3))
    # save results
    with open(f"results/{model_id}_{model_type}.pkl", "wb") as f:
        pickle.dump(cv_test, f)

In [6]:
cross_validation_binary_tree_us("model_1a","binary_tree")

model_1a binary_tree
Recall:  0.742
Roc:  0.817
Accuracy:  0.749


In [7]:
cross_validation_binary_tree_us("model_2a","binary_tree")

model_2a binary_tree
Recall:  0.465
Roc:  0.689
Accuracy:  0.629


In [8]:
cross_validation_binary_tree_us("model_1b","binary_tree")

model_1b binary_tree
Recall:  0.818
Roc:  0.856
Accuracy:  0.809


In [9]:
cross_validation_binary_tree_us("model_2b","binary_tree")

model_2b binary_tree
Recall:  0.497
Roc:  0.663
Accuracy:  0.578
